In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
from pathlib import Path

In [2]:
from google.colab import files
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
path=Path('/content/drive/MyDrive/apache spam')

In [4]:
for folder in path.iterdir():
  print(folder)

/content/drive/MyDrive/apache spam/20030228_spam
/content/drive/MyDrive/apache spam/20030228_easy_ham


In [5]:
path_spam=Path('/content/drive/MyDrive/apache spam/20030228_spam/spam')
path_ham=Path('/content/drive/MyDrive/apache spam/20030228_easy_ham/easy_ham')

In [6]:
for file in path_ham.iterdir():
  with open(file,encoding='utf8') as f:
    content=f.read()
    break

In [7]:
print(content)

From spamassassin-commits-admin@lists.sourceforge.net  Thu Aug 29 11:07:34 2002
Return-Path: <spamassassin-commits-admin@example.sourceforge.net>
Delivered-To: yyyy@localhost.netnoteinc.com
Received: from localhost (localhost [127.0.0.1])
	by phobos.labs.netnoteinc.com (Postfix) with ESMTP id E938744157
	for <jm@localhost>; Thu, 29 Aug 2002 06:05:49 -0400 (EDT)
Received: from phobos [127.0.0.1]
	by localhost with IMAP (fetchmail-5.9.0)
	for jm@localhost (single-drop); Thu, 29 Aug 2002 11:05:49 +0100 (IST)
Received: from usw-sf-list2.sourceforge.net (usw-sf-fw2.sourceforge.net
    [216.136.171.252]) by dogma.slashnull.org (8.11.6/8.11.6) with ESMTP id
    g7SNTpZ15202 for <jm@jmason.org>; Thu, 29 Aug 2002 00:29:52 +0100
Received: from usw-sf-list1-b.sourceforge.net ([10.3.1.13]
    helo=usw-sf-list1.sourceforge.net) by usw-sf-list2.sourceforge.net with
    esmtp (Exim 3.31-VA-mm2 #1 (Debian)) id 17kCGA-00016L-00; Wed,
    28 Aug 2002 16:30:02 -0700
Received: from usw-sf-sshgate.sourcefo

In [8]:
import tarfile

In [9]:
from pathlib import Path

In [10]:
import urllib

In [11]:
def fetch_spam_data():
  spam_root="http://spamassassin.apache.org/old/publiccorpus/"
  ham_url=spam_root+'20030228_easy_ham.tar.bz2'
  spam_url=spam_root+"20030228_spam.tar.bz2"
  spam_path=Path()/'datasets'/'spams'
  spam_path.mkdir(parents=True,exist_ok=True)
  for dir_name,tar_name,url in (('easy_ham','ham',ham_url),('spam','spam',spam_url)):
    if not (spam_path/dir_name).is_dir():
      path=(spam_path/tar_name).with_suffix('.tar.bz2')
      print('Downloading',path)
      urllib.request.urlretrieve(url,path)
      tar_bz2_file=tarfile.open(path)
      tar_bz2_file.extractall(path=spam_path,filter='data')
      tar_bz2_file.close()
  return [spam_path/dir_name for dir_name in('easy_ham','spam')]

In [12]:
ham_dir,spam_dir=fetch_spam_data()

In [13]:
len_checker={}

In [14]:
from collections import Counter

In [15]:
for f in ham_dir.iterdir():
  len_checker.setdefault(len(f.name),[]).append(1)

In [16]:
for size,counting in len_checker.items():
  print(size,len(counting))

38 2500
4 1


In [17]:
Counter(len(file.name) for file in spam_dir.iterdir())

Counter({38: 500, 4: 1})

In [18]:
ham_filenames=[f for f in sorted(ham_dir.iterdir()) if len(f.name)>20]
spam_filenames=[f for f in sorted(spam_dir.iterdir()) if len(f.name)>20]

In [19]:
import email
import email.policy

def load_email(filepath):
  with open(filepath,'rb') as f:
    return email.parser.BytesParser(policy=email.policy.default).parse(f)

In [20]:
ham_emails=[load_email(file) for file in ham_filenames]
spam_emails=[load_email(file) for file in spam_filenames]

In [21]:
print(ham_emails[10].get_content().strip())

Hello, have you seen and discussed this article and his approach?

Thank you

http://www.paulgraham.com/spam.html
-- "Hell, there are no rules here-- we're trying to accomplish something."
-- Thomas Alva Edison




-------------------------------------------------------
This sf.net email is sponsored by: OSDN - Tired of that same old
cell phone?  Get a new here for FREE!
https://www.inphonic.com/r.asp?r=sourceforge1&refcode1=vs3390
_______________________________________________
Spamassassin-devel mailing list
Spamassassin-devel@lists.sourceforge.net
https://lists.sourceforge.net/lists/listinfo/spamassassin-devel


In [22]:
print(spam_emails[1].get_content().strip())

1) Fight The Risk of Cancer!
http://www.adclick.ws/p.cfm?o=315&s=pk007

2) Slim Down - Guaranteed to lose 10-12 lbs in 30 days
http://www.adclick.ws/p.cfm?o=249&s=pk007

3) Get the Child Support You Deserve - Free Legal Advice
http://www.adclick.ws/p.cfm?o=245&s=pk002

4) Join the Web's Fastest Growing Singles Community
http://www.adclick.ws/p.cfm?o=259&s=pk007

5) Start Your Private Photo Album Online!
http://www.adclick.ws/p.cfm?o=283&s=pk007

Have a Wonderful Day,
Offer Manager
PrizeMama













If you wish to leave this list please use the link below.
http://www.qves.com/trim/?ilug@linux.ie%7C17%7C114258


-- 
Irish Linux Users' Group: ilug@linux.ie
http://www.linux.ie/mailman/listinfo/ilug for (un)subscription information.
List maintainer: listmaster@linux.ie


In [23]:
def get_email_structure(email):
  if isinstance(email,str):
    return email
  payload=email.get_payload()
  if isinstance(payload,list):
    multipart=','.join([get_email_structure(parts) for parts in payload])
    return f"multipart({multipart})"
  return email.get_content_type()

In [24]:
from collections import Counter

In [25]:
def structure_counter(emails):
  structures={}
  for i,email in enumerate(emails):
    structure=get_email_structure(email)
    structures.setdefault(structure,[]).append(i)
  return structures

In [26]:
result_spam=structure_counter(spam_emails)

In [27]:
for key, value in result_spam.items():
    print(key, ":", value)

text/html : [0, 7, 9, 11, 13, 14, 24, 27, 28, 29, 34, 35, 36, 41, 43, 45, 48, 49, 51, 52, 54, 57, 58, 60, 65, 66, 82, 85, 87, 90, 93, 98, 100, 103, 106, 108, 113, 118, 120, 122, 124, 129, 132, 133, 135, 138, 142, 145, 146, 147, 148, 155, 160, 161, 163, 164, 165, 166, 167, 171, 172, 173, 174, 175, 176, 178, 180, 182, 187, 188, 189, 194, 195, 198, 199, 200, 201, 203, 204, 210, 211, 212, 214, 216, 220, 221, 230, 232, 243, 245, 250, 258, 265, 267, 273, 275, 280, 281, 283, 284, 285, 287, 289, 294, 295, 296, 299, 301, 303, 305, 309, 312, 313, 320, 321, 328, 330, 335, 344, 347, 349, 350, 352, 354, 356, 363, 367, 368, 373, 374, 375, 377, 379, 382, 384, 386, 390, 394, 395, 396, 397, 401, 404, 405, 406, 407, 411, 413, 415, 416, 417, 423, 424, 429, 432, 436, 440, 442, 447, 448, 449, 450, 452, 454, 455, 456, 460, 469, 470, 472, 473, 475, 476, 482, 486, 487, 490, 491, 492, 495, 496, 498, 499]
text/plain : [1, 2, 3, 4, 5, 6, 8, 10, 12, 15, 16, 17, 18, 19, 20, 25, 26, 30, 31, 32, 33, 39, 40, 42, 44, 

In [28]:
[email.get_payload() for email in spam_emails[238].get_payload()]

[[<email.message.EmailMessage at 0x7def9133b570>]]

In [29]:
[vidyut_sandesh for vidyut_sandesh in spam_emails[125].get_payload()[1]]

['Content-Type', 'Content-Disposition']

In [30]:
print( spam_emails[125])

Return-Path: <eyeey@keromail.com>
Delivered-To: zzzz@localhost.spamassassin.taint.org
Received: from localhost (localhost [127.0.0.1])
	by phobos.labs.spamassassin.taint.org (Postfix) with ESMTP id 7013744155
	for <zzzz@localhost>; Wed, 28 Aug 2002 05:42:54 -0400 (EDT)
Received: from mail.webnote.net [193.120.211.219]
	by localhost with POP3 (fetchmail-5.9.0)
	for zzzz@localhost (single-drop); Wed, 28 Aug 2002 10:42:54 +0100 (IST)
Received: from SERVER2 (mail.nomioan.gr [195.167.25.66])
	by webnote.net (8.9.3/8.9.3) with ESMTP id UAA00879
	for <zzzz@spamassassin.taint.org>; Tue, 27 Aug 2002 20:57:14 +0100
From: eyeey@keromail.com
Received: from keromail-com.mr.outblaze.com (203.117.141.101) by SERVER2 with
 SMTP; Tue, 27 Aug 2002 22:33:42 +0300
Message-ID: <000026a91900$00002e3e$000025db@m208.hk.renren.com>
To: <customerapparelu@webnote.net>
Subject: Memory loss?
Date: Tue, 27 Aug 2002 13:41:46 -1600
MIME-Version: 1.0
Reply-To: discovery9@gandabacha.com
Content-Type: multipart/mixed; b

in 125, it has multipart of text/html and text/plain. now in this- text/plain contains only 4 letters.<br>
get_content wont work when there is a multipart

In [31]:
print(spam_emails[466])

Return-Path: <InkjetDeals@acsmsupplies.com>
Delivered-To: zzzz@localhost.spamassassin.taint.org
Received: from localhost (jalapeno [127.0.0.1])
	by zzzzason.org (Postfix) with ESMTP id BF27916F03
	for <zzzz@localhost>; Thu, 26 Sep 2002 11:11:52 +0100 (IST)
Received: from jalapeno [127.0.0.1]
	by localhost with IMAP (fetchmail-5.9.0)
	for zzzz@localhost (single-drop); Thu, 26 Sep 2002 11:11:52 +0100 (IST)
Received: from acsmsupplies.com (host02.acsm.synergy-networks.com
    [64.200.6.66] (may be forged)) by dogma.slashnull.org (8.11.6/8.11.6) with
    SMTP id g8PIPHC08718 for <webmaster@efi.ie>; Wed, 25 Sep 2002 19:25:17
    +0100
Message-Id: <200209251825.g8PIPHC08718@dogma.slashnull.org>
From: "Affordable Computer Supply" <InkjetDeals@acsmsupplies.com>
To: <webmaster@efi.ie>
Subject: Printer Cartridges as low as $1.21 each!
Sender: "Affordable Computer Supply" <InkjetDeals@acsmsupplies.com>
MIME-Version: 1.0
Date: Wed, 25 Sep 2002 14:29:23 -0400
Reply-To: "Affordable Computer Supply" 

In [32]:
spam_emails[466].get_payload()

'This is a multipart MIME message.\n\n--= Multipart Boundary 0925021429\nContent-Type: text/plain;\n\tcharset="ISO-8859-1"\nContent-Transfer-Encoding: 8bit\n\n<html>\n\n<head>\n</head>\n\n<body>\n\n<p align="center"><span style="font-size:7.5pt;font-family:Arial;color:black">You\nare receiving this email because you opted-in to receive special offers from\nOptinDeals through one of our marketing partners. If you feel you have received\nthis email in error or do not wish to receive additional special offers, please\nreply to this email with the word &quot;remove&quot; in the subject line or\nfollow the unsubscribe instructions below.<o:p>\n</o:p>\n</span></p>\n<p><a href="http://www.inkjetrus.com/customer!.html"><img border="0" \nsrc="http://www.inkjetrus.com/clip%20art/toper.jpg" width="800" height="100"></a>\n</p>\n<div align="center">\n  <strong><font face="Arial" size="6">Incredible Deals on Inkjet Cartridges!\n</font></strong>\n</div>\n<div align="center">\n  <strong><font face="Ar

In [33]:
spam_emails[0].get_payload()

'<!DOCTYPE HTML PUBLIC "-//W3C//DTD HTML 4.0 Transitional//EN">\n<HTML><HEAD>\n<META content=3D"text/html; charset=3Dwindows-1252" http-equiv=3DContent-T=\nype>\n<META content=3D"MSHTML 5.00.2314.1000" name=3DGENERATOR></HEAD>\n<BODY><!-- Inserted by Calypso -->\n<TABLE border=3D0 cellPadding=3D0 cellSpacing=3D2 id=3D_CalyPrintHeader_ r=\nules=3Dnone \nstyle=3D"COLOR: black; DISPLAY: none" width=3D"100%">\n  <TBODY>\n  <TR>\n    <TD colSpan=3D3>\n      <HR color=3Dblack noShade SIZE=3D1>\n    </TD></TR></TD></TR>\n  <TR>\n    <TD colSpan=3D3>\n      <HR color=3Dblack noShade SIZE=3D1>\n    </TD></TR></TBODY></TABLE><!-- End Calypso --><!-- Inserted by Calypso=\n --><FONT \ncolor=3D#000000 face=3DVERDANA,ARIAL,HELVETICA size=3D-2><BR></FONT></TD><=\n/TR></TABLE><!-- End Calypso --><FONT color=3D#ff0000 \nface=3D"Copperplate Gothic Bold" size=3D5 PTSIZE=3D"10">\n<CENTER>Save up to 70% on Life Insurance.</CENTER></FONT><FONT color=3D#ff=\n0000 \nface=3D"Copperplate Gothic Bold" size=3D5 P

In [34]:
spam_emails[0].get_content()

'<!DOCTYPE HTML PUBLIC "-//W3C//DTD HTML 4.0 Transitional//EN">\n<HTML><HEAD>\n<META content="text/html; charset=windows-1252" http-equiv=Content-Type>\n<META content="MSHTML 5.00.2314.1000" name=GENERATOR></HEAD>\n<BODY><!-- Inserted by Calypso -->\n<TABLE border=0 cellPadding=0 cellSpacing=2 id=_CalyPrintHeader_ rules=none \nstyle="COLOR: black; DISPLAY: none" width="100%">\n  <TBODY>\n  <TR>\n    <TD colSpan=3>\n      <HR color=black noShade SIZE=1>\n    </TD></TR></TD></TR>\n  <TR>\n    <TD colSpan=3>\n      <HR color=black noShade SIZE=1>\n    </TD></TR></TBODY></TABLE><!-- End Calypso --><!-- Inserted by Calypso --><FONT \ncolor=#000000 face=VERDANA,ARIAL,HELVETICA size=-2><BR></FONT></TD></TR></TABLE><!-- End Calypso --><FONT color=#ff0000 \nface="Copperplate Gothic Bold" size=5 PTSIZE="10">\n<CENTER>Save up to 70% on Life Insurance.</CENTER></FONT><FONT color=#ff0000 \nface="Copperplate Gothic Bold" size=5 PTSIZE="10">\n<CENTER>Why Spend More Than You Have To?\n<CENTER><FONT co

In [35]:
result_ham=structure_counter(ham_emails)

In [36]:
for key,values in result_ham.items():
  print(key,':',values)

text/plain : [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 63, 64, 65, 67, 68, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116, 117, 118, 119, 120, 121, 122, 123, 124, 125, 126, 127, 128, 129, 130, 131, 132, 133, 134, 135, 136, 137, 138, 139, 140, 141, 142, 143, 144, 145, 146, 147, 148, 149, 150, 151, 152, 153, 154, 155, 156, 157, 158, 159, 160, 161, 162, 163, 164, 166, 167, 168, 169, 170, 171, 172, 173, 174, 175, 176, 177, 178, 179, 180, 181, 182, 183, 184, 185, 186, 187, 188, 189, 190, 191, 192, 193, 194, 195, 196, 197, 198, 199, 200, 201, 202, 203, 204, 205, 206, 207, 208, 209, 210, 211, 212, 213, 214, 215, 216, 217, 218, 219, 220, 221, 222, 223, 2

In [37]:
for sandesh in ham_emails[13].get_payload():
  print(sandesh)
  print()

Content-Type: text/plain; charset=us-ascii

> From:  Chris Garrigues <cwg-exmh@DeepEddy.Com>
> Date:  Wed, 21 Aug 2002 10:40:39 -0500
>
> > From:  Chris Garrigues <cwg-exmh@DeepEddy.Com>
> > Date:  Wed, 21 Aug 2002 10:17:45 -0500
> >
> > Ouch...I'll get right on it.
> > 
> > > From:  Robert Elz <kre@munnari.OZ.AU>
> > > Date:  Wed, 21 Aug 2002 19:30:01 +0700
> > >
> > > Any chance of having that lengthen instead?   I like all my exmh stuff
> > > in nice columns (fits the display better).   That is, I use the detache
> d
> > > folder list, one column.   The main exmh window takes up full screen,
> > > top to bottom, but less than half the width, etc...
> 
> I thought about that.  The first order approximation would be to just add 
> using pack .... -side top instead of pack ... -side left, however, since their 
> each a different width, it would look funny.

I've done this.  It's not as pretty as I think it should be, but it works.  
I'm going to leave the cosmetic issues to others.  Wh

In [38]:
print(ham_emails[13])

Return-Path: <exmh-workers-admin@spamassassin.taint.org>
Delivered-To: zzzz@localhost.netnoteinc.com
Received: from localhost (localhost [127.0.0.1])
	by phobos.labs.netnoteinc.com (Postfix) with ESMTP id 50AF343F9B
	for <zzzz@localhost>; Thu, 22 Aug 2002 11:37:35 -0400 (EDT)
Received: from phobos [127.0.0.1]
	by localhost with IMAP (fetchmail-5.9.0)
	for zzzz@localhost (single-drop); Thu, 22 Aug 2002 16:37:35 +0100 (IST)
Received: from listman.spamassassin.taint.org (listman.spamassassin.taint.org
 [66.187.233.211]) by    dogma.slashnull.org (8.11.6/8.11.6) with ESMTP id
 g7MFZLZ12577 for    <zzzz-exmh@spamassassin.taint.org>; Thu, 22 Aug 2002
 16:35:22 +0100
Received: from listman.spamassassin.taint.org (localhost.localdomain
 [127.0.0.1]) by    listman.redhat.com (Postfix) with ESMTP id 51F5140D92;
 Thu, 22 Aug 2002    11:35:26 -0400 (EDT)
Delivered-To: exmh-workers@listman.spamassassin.taint.org
Received: from int-mx1.corp.spamassassin.taint.org
 (int-mx1.corp.spamassassin.taint.or

In [39]:
for sandesh in ham_emails[66].get_payload():
  print(sandesh)
  print()

Content-Type: text/plain; charset="us-ascii"
Content-Transfer-Encoding: 7bit

........... with our telecoms partner Bumblebee !

Don't get ripped off by expensive hotel, payphone and mobile charges. 
SAVE, SAVE, SAVE on international calls with Ryanair's phone partner. 
************************************************************************
*********

You'll save up to 70% on international phone calls when you use our 
online phone card.  You can use the card from any phone in any country 
you visit and you won't have to worry about high phone charges 
when you call home or the office.

Buying a card couldn't be easier and it's totally secure.  Simply go to
http://www.bumblebeecommunications.com/lowcostcalls/  
 to avail of this special offer for Ryanair customers.

It's another great deal from Ryanair and our online phone 
partner, Bumblebee Communications. 

 








E-MAIL DISCLAIMER

This e-mail and any files and attachments transmitted with it 
are confidential and may be legall

In [40]:
counter_spam={key:len(values) for key,values in result_spam.items()}
for key, value in counter_spam.items():
  print(key,":",value)

text/html : 183
text/plain : 218
multipart(text/plain,application/octet-stream) : 1
multipart(text/html) : 20
multipart(text/plain,text/html) : 45
multipart(text/plain) : 19
multipart(text/html,text/plain) : 1
multipart(text/html,application/octet-stream) : 2
multipart(multipart(text/html)) : 5
multipart(text/plain,image/jpeg) : 3
multipart(multipart(text/html),application/octet-stream,image/jpeg) : 1
multipart(multipart(text/plain,text/html),image/gif) : 1
multipart/alternative : 1


In [41]:
counter_ham={key:len(values) for key,values in result_ham.items()}
for key, value in counter_ham.items():
  print(key,":",value)

text/plain : 2408
multipart(text/plain,application/pgp-signature) : 66
multipart(text/plain,text/html) : 8
multipart(text/plain,text/enriched) : 1
multipart(text/plain,application/ms-tnef,text/plain) : 1
multipart(text/plain) : 3
multipart(text/plain,application/octet-stream) : 2
multipart(text/plain,text/plain) : 4
multipart(multipart(text/plain,text/plain,text/plain),application/pgp-signature) : 1
multipart(text/plain,video/mng) : 1
multipart(text/plain,multipart(text/plain)) : 1
multipart(text/plain,application/x-pkcs7-signature) : 1
multipart(text/plain,multipart(text/plain,text/plain),text/rfc822-headers) : 1
multipart(text/plain,multipart(text/plain,text/plain),multipart(multipart(text/plain,application/x-pkcs7-signature))) : 1
multipart(text/plain,application/x-java-applet) : 1


In [42]:
for header, value in spam_emails[0].items():
  print(header,':',value)

Return-Path : <12a1mailbot1@web.de>
Delivered-To : zzzz@localhost.spamassassin.taint.org
Received : from localhost (localhost [127.0.0.1])	by phobos.labs.spamassassin.taint.org (Postfix) with ESMTP id 136B943C32	for <zzzz@localhost>; Thu, 22 Aug 2002 08:17:21 -0400 (EDT)
Received : from mail.webnote.net [193.120.211.219]	by localhost with POP3 (fetchmail-5.9.0)	for zzzz@localhost (single-drop); Thu, 22 Aug 2002 13:17:21 +0100 (IST)
Received : from dd_it7 ([210.97.77.167])	by webnote.net (8.9.3/8.9.3) with ESMTP id NAA04623	for <zzzz@spamassassin.taint.org>; Thu, 22 Aug 2002 13:09:41 +0100
From : 12a1mailbot1@web.de
Received : from r-smtp.korea.com - 203.122.2.197 by dd_it7  with Microsoft SMTPSVC(5.5.1775.675.6);	 Sat, 24 Aug 2002 09:42:10 +0900
To : dcek1a1@netsgo.com
Subject : Life Insurance - Why Pay More?
Date : Wed, 21 Aug 2002 20:31:57 -1600
MIME-Version : 1.0
Message-ID : <0103c1042001882DD_IT7@dd_it7>
Content-Type : text/html; charset="iso-8859-1"
Content-Transfer-Encoding : qu

In [43]:
spam_emails[0]['Subject']

'Life Insurance - Why Pay More?'

In [44]:
spam_emails[0].get_content_type()

'text/html'

In [45]:
import numpy as np
from sklearn.model_selection import train_test_split

In [46]:
X=np.array(ham_emails+spam_emails,dtype=object)
y=np.array([0]*len(ham_emails)+[1]*len(spam_emails))

In [47]:
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,stratify=y,random_state=42)

In [48]:
import re
from html import unescape

In [120]:
def html_to_plain(html):
  text=re.sub(r'<head.*?>.*?</head>','',html,flags=re.S|re.I)
  text=re.sub(r'<a\s.*?>','HYPERLINK',text,flags=re.S|re.I)
  text=re.sub(r'<.*?>','',text,flags=re.S)
  text=re.sub(r'(\s*\n)+','\n',text,flags=re.S)
  return unescape(text)

In [114]:
html_spam_emails=[email for email in X_train[y_train==1] if get_email_structure(email)=='text/html']

In [118]:
sample_email=html_spam_emails[70]
print(sample_email.get_content())

iieo
<html>
<head>
<title>Mortgage companies make you wait</title>
</head>

<body>

<div align="center">
  <center>
  <table height="1" cellSpacing="0" width="540" border="0" style="border-collapse: collapse" bordercolor="#111111" cellpadding="0">
    <tr>
      <td colSpan="2">
      <p align="center"><font face="Tahoma" color="#008000"><big><strong>
      Mortgage companies make you wait...They Demand to Interview you...They <u>
      Intimidate</u> you...They <u>Humiliate</u> you...And All of That is While 
      They Decide If They Even Want to Do Business With You...</strong></big></p>
      <p align="center"><font color="#000000" size="5"><strong>We Turn the 
      Tables on Them...<br>
      Now, You're In Charge</strong></font></p>
      <p align="center"><font color="#000000"><small><b>Just Fill Out Our Simple 
      Form and They Will Have to Compete For Your Business...</b></small><br>
      <br>
      <big><big><strong>
      <a target="_NEW" href="http://208.251.89.19/mort

In [123]:
print(html_to_plain(sample_email.get_content()))

iieo
      Mortgage companies make you wait...They Demand to Interview you...They
      Intimidate you...They Humiliate you...And All of That is While
      They Decide If They Even Want to Do Business With You...
      We Turn the
      Tables on Them...
      Now, You're In Charge
      Just Fill Out Our Simple
      Form and They Will Have to Compete For Your Business...
      HYPERLINK
      CLICK HERE FOR THE FORM
      We have hundreds of loan
      programs, including:
        Refinance
        Debt Consolidation
        Home Improvement
        Second Mortgages
        Cash Out
        Home Equity Loan
          You can save Thousands Of Dollars
          over the course of your loan with just a 1/4 of 1% Drop in your rate!
 
 
      HYPERLINK
      CLICK HERE FOR THE FORM
      You
      will be contacted with an offer within 24 hours from the time you fill out
      the form!
Please know that we do not want to send you information regarding our special offers if you do not wi

In [129]:
from email.message import EmailMessage

msg=spam_emails[123]

for part in msg.walk():
    content_type = part.get_content_type()
    filename = part.get_filename()

    print(f"Content Type: {content_type}, Filename: {filename}")
    print(part)
    print()

Content Type: multipart/alternative, Filename: None
Return-Path: <iman@s3.serveimage.com>
Delivered-To: zzzz@localhost.spamassassin.taint.org
Received: from localhost (localhost [127.0.0.1])
	by phobos.labs.spamassassin.taint.org (Postfix) with ESMTP id 8D25743F99
	for <zzzz@localhost>; Wed, 28 Aug 2002 05:42:48 -0400 (EDT)
Received: from mail.webnote.net [193.120.211.219]
	by localhost with POP3 (fetchmail-5.9.0)
	for zzzz@localhost (single-drop); Wed, 28 Aug 2002 10:42:48 +0100 (IST)
Received: from email.qves.com (email5.qves.net [209.63.151.251] (may be
 forged))	by webnote.net (8.9.3/8.9.3) with ESMTP id TAA00664	for
 <zzzz@spamassassin.taint.org>; Tue, 27 Aug 2002 19:49:07 +0100
Received: from qvp0088 ([169.254.6.9]) by email.qves.com with Microsoft
 SMTPSVC(5.0.2195.2966);	 Tue, 27 Aug 2002 12:48:32 -0600
From: "How Slim" <iman@s3.serveimage.com>
To: <zzzz@spamassassin.taint.org>
Subject: Guaranteed to lose 10-12 lbs in 30 days with Slim Patch
                    10.24
Date: Tue,

In [143]:
def email_to_text(email):
  html_parts=[]
  for part in email.walk():
    ctype=part.get_content_type()
    if not ctype in ('text/plain','text/html'):
      continue
    try:
      content=ctype.get_content()
    except:
      content=str(part.get_payload())
    if ctype=='text/plain':
      return content
    else:
      html_parts.append(content)
  if html_parts:
    return "\n".join(html_to_plain(html) for html in html_parts)

In [133]:
print(email_to_text(sample_email)[:100])

iieo
      Mortgage companies make you wait...They Demand to Interview you...Th=
ey
      Intimidate


In [157]:
import nltk
stemmer=nltk.PorterStemmer()

In [138]:
 pip install urlextract

In [141]:
import urlextract

url_extractor=urlextract.URLExtract()
some_text='ad.com and many other things https://runebook.dev/en/docs/python/library/email.message/email.message.EmailMessage.walk'
print(url_extractor.find_urls(some_text))

['ad.com', 'https://runebook.dev/en/docs/python/library/email.message/email.message.EmailMessage.walk']


In [142]:
from sklearn.base import BaseEstimator, TransformerMixin

In [160]:
class EmailToWordCounterTransformer(BaseEstimator,TransformerMixin):
  def __init__(self,strip_headers=True,lower_case=True,remove_punctuation=True,
               replace_urls=True,replace_numbers=True,stemming=True):
    self.strip_headers=strip_headers
    self.lower_case=lower_case
    self.remove_punctuation=remove_punctuation
    self.replace_urls=replace_urls
    self.replace_numbers=replace_numbers
    self.stemming=stemming
  def fit(self,X,y=None):
    return self
  def transform(self,X,y=None):
    X_transformed=[]
    for email in X:
      text=email_to_text(email) or ""
      if not self.strip_headers:
        headers='\n'.join(f"{key}:{value}" for key, value in email.items())
        text=headers + "\n" + text
      if self.lower_case:
        text=text.lower()
      if self.replace_urls and url_extractor is not None:
        urls=list(set(url_extractor.find_urls(text)))
        urls.sort(key=lambda url:len(url),reverse=True)
        for url in urls:
          text=text.replace(url,'URL')
      if self.replace_numbers:
        text=re.sub(r'\d+(?:\.\d*)?(?:[eE][+-]?\d+)?','NUMBER',text)
      if self.remove_punctuation:
        text=re.sub(r'\W+',' ', text)
      word_counters=Counter(text.split())
      if self.stemming and stemmer is not None:
        stemmed_word_counts=Counter()
        for word, count in word_counters.items():
          stemmed_word=stemmer.stem(word)
          stemmed_word_counts[stemmed_word]+=count
        word_counters=stemmed_word_counts

      X_transformed.append(word_counters)
    return np.array(X_transformed)


In [171]:
X_few=X_train[:4]
X_few_transformed=EmailToWordCounterTransformer().fit_transform(X_few)

In [167]:
print(X_train[0].get_content()[:100])

>>>>> "R" == Russell Turpin <deafbox@hotmail.com> writes:

    R> Things aren't all that bad. I reme


In [169]:
from scipy.sparse import csr_matrix

In [229]:
X_few_transformed

array([Counter({'in': 6, 'of': 5, 'and': 5, 'the': 5, 'you': 5, 'r': 4, 'that': 3, 'i': 3, 'a': 3, 'eateri': 3, 'toronto': 3, 'are': 3, 'not': 3, 'com': 2, 't': 2, 'as': 2, 'have': 2, 'varieti': 2, 'good': 2, 'local': 2, 'strip': 2, 'but': 2, 'neighbourhood': 2, 'more': 2, 'than': 2, 'or': 2, 'there': 2, 'like': 2, 'can': 2, 'at': 2, 'doe': 2, 'live': 2, 'commun': 2, 'russel': 1, 'turpin': 1, 'deafbox': 1, 'hotmail': 1, 'write': 1, 'thing': 1, 'aren': 1, 'all': 1, 'bad': 1, 'rememb': 1, 'vancouv': 1, 'broad': 1, 'joint': 1, 'haven': 1, 'been': 1, 'to': 1, 'van': 1, 'year': 1, 'do': 1, 'know': 1, 'outsid': 1, 'small': 1, 'deepli': 1, 'ethnic': 1, 'if': 1, 'stray': 1, 'number': 1, 'feet': 1, 'from': 1, 'wellesley': 1, 'jarvi': 1, 'queen': 1, 'spadina': 1, 're': 1, 'doughnutland': 1, 's': 1, 'far': 1, 'eaton': 1, 'centr': 1, 'quaint': 1, 'cafe': 1, 'mccaul': 1, 'north': 1, 'dunda': 1, 'rare': 1, 'littl': 1, 'brim': 1, 'with': 1, 'lunchtim': 1, 'traffic': 1, 'still': 1, 'find': 1, 'seat': 

In [230]:
class WordCounterToVectorTransformer(BaseEstimator,TransformerMixin):
  def __init__(self,vocabulary_size=1000):
    self.vocabulary_size=vocabulary_size
  def fit(self,X,y=None):
    total_word=Counter()
    for word_count in X:
      for word, count in word_count.items():
        total_word[word]+=min(count,10)
    vocabulary=total_word.most_common()[:self.vocabulary_size]
    self.count_of_words=vocabulary
    self.vocabulary_={word:i+1 for i,(word,_) in enumerate(vocabulary)}
    return self
  def transform(self,X,y=None):
    rows=[]
    cols=[]
    data=[]
    for row,word_count in enumerate(X):
      for word,count in word_count.items():
        rows.append(row)
        cols.append(self.vocabulary_.get(word,0))
        data.append(count)
    return csr_matrix((data,(rows,cols)),shape=(len(X),self.vocabulary_size+1))

In [231]:
vocab_size=WordCounterToVectorTransformer(vocabulary_size=10)#an instance of class, this will contain necessary attributes.
RESULTS=vocab_size.fit_transform(X_few_transformed)

In [232]:
RESULTS.toarray()#scipy.csr.matrix

array([[155,   1,   5,   5,   1,   6,   5,   5,   1,   0,   3],
       [107,  18,   6,   2,  10,   4,   0,   3,   4,   4,   0],
       [ 70,   2,   1,   1,   1,   1,   3,   1,   2,   1,   1],
       [ 78,  23,   3,   4,   0,   0,   1,   0,   1,   3,   3]])

In [228]:
print(X_few[0].get_content())

>>>>> "R" == Russell Turpin <deafbox@hotmail.com> writes:

    R> Things aren't all that bad. I remember Vancouver as having a
    R> broad variety of good, local eateries.  And Toronto as having a
    R> variety of good, local strip joints. ;-) 

I haven't been to Van in years, but I do know that in Toronto, outside
of the small deeply ethnic neighbourhoods, if you stray more than 50
feet from Wellesley and Jarvis or Queen and Spadina, you're in
doughnutland; there's far more like the eateries in the Eaton Centre
than there are quaint cafes like McCaul north of Dundas, and the rare
little eateries are not brimming with lunchtime traffic (you can still
find a seat at noon at Village by the Grange)

When most of Toronto does not live in those neighbourhoods but instead
lives in Scarborough, Malton, Mississauga and Markham (franchisevilles), you
will quickly see that one or two trendy strips does not save an entire
nation.


-- 
Gary Lawrence Murphy <garym@teledyn.com> TeleDynamics Commu

In [233]:
vocab_size.vocabulary_

{'number': 1,
 'and': 2,
 'the': 3,
 'url': 4,
 'in': 5,
 'of': 6,
 'you': 7,
 'to': 8,
 'on': 9,
 'that': 10}

In [234]:
vocab_size.count_of_words

[('number', 23),
 ('and', 15),
 ('the', 12),
 ('url', 12),
 ('in', 11),
 ('of', 9),
 ('you', 9),
 ('to', 8),
 ('on', 8),
 ('that', 7)]

In [255]:
X_100=X_train[-100:]
X_100_transformed=EmailToWordCounterTransformer().fit_transform(X_100)

In [258]:
vocab_size=WordCounterToVectorTransformer(vocabulary_size=15)
X_100_vector=vocab_size.fit_transform(X_100_transformed)

In [269]:
X_100_vector.toarray()[:10].tolist()

[[79, 0, 3, 2, 2, 2, 2, 0, 0, 3, 1, 3, 2, 1, 1, 1],
 [84, 4, 4, 0, 3, 0, 0, 1, 4, 3, 2, 0, 3, 0, 0, 0],
 [59, 9, 2, 1, 3, 3, 1, 1, 5, 3, 0, 2, 0, 1, 1, 1],
 [236, 3, 10, 8, 10, 8, 7, 1, 0, 4, 14, 1, 5, 3, 4, 5],
 [662, 38, 75, 13, 15, 25, 43, 3, 2, 6, 11, 10, 11, 0, 21, 9],
 [22, 0, 0, 0, 0, 0, 0, 0, 2, 0, 0, 1, 0, 1, 0, 0],
 [165, 19, 11, 7, 10, 9, 3, 1, 0, 0, 0, 9, 2, 3, 4, 2],
 [39, 0, 2, 2, 0, 0, 0, 0, 2, 0, 0, 0, 1, 0, 1, 0],
 [57, 1, 4, 2, 0, 0, 0, 1, 2, 0, 2, 1, 1, 1, 0, 0],
 [307, 10, 22, 10, 7, 14, 15, 11, 0, 1, 3, 5, 9, 6, 6, 2]]

In [273]:
print(url_extractor.find_urls(X_train[-97].get_content()))

[]


In [262]:
vocab_size.count_of_words

[('number', 557),
 ('the', 486),
 ('to', 383),
 ('a', 335),
 ('and', 296),
 ('of', 258),
 ('i', 234),
 ('url', 228),
 ('it', 213),
 ('is', 209),
 ('for', 198),
 ('in', 195),
 ('you', 171),
 ('that', 171),
 ('on', 155)]

In [264]:
print(vocab_size.vocabulary_)

{'number': 1, 'the': 2, 'to': 3, 'a': 4, 'and': 5, 'of': 6, 'i': 7, 'url': 8, 'it': 9, 'is': 10, 'for': 11, 'in': 12, 'you': 13, 'that': 14, 'on': 15}


In [278]:
from sklearn.pipeline import Pipeline,make_pipeline

In [275]:
preprocess_pipeline=Pipeline([
    ("email_to_word_count",EmailToWordCounterTransformer()),
    ("word_count_to_vector",WordCounterToVectorTransformer())
])

In [276]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score

In [279]:
log_clf=LogisticRegression(max_iter=1000,random_state=42)
log_pipeline=make_pipeline(preprocess_pipeline,log_clf)

In [280]:
score=cross_val_score(log_pipeline,X_train,y_train,cv=3)
score.mean()

np.float64(0.9829166666666667)

In [281]:
from sklearn.metrics import accuracy_score,precision_score,recall_score,f1_score,classification_report,ConfusionMatrixDisplay

In [284]:
X_train_transformed=preprocess_pipeline.fit_transform(X_train)
X_test_transformed=preprocess_pipeline.fit_transform(X_test)

In [285]:
log_clf=LogisticRegression(max_iter=1000,random_state=42)
log_clf.fit(X_train_transformed,y_train)

LogisticRegression(max_iter=1000, random_state=42)

In [286]:
y_pred=log_clf.predict(X_test_transformed)

In [302]:
print(f"Accuracy : {accuracy_score(y_test,y_pred):.3%}")
print(f"Precision : {precision_score(y_test,y_pred):.3%}")
print(f"Recall : {recall_score(y_test,y_pred):.3%}")
print(f"F-1 score : {f1_score(y_test,y_pred):.3%}")
print("Classification Report:")
print(f"{classification_report(y_test,y_pred)}")

Accuracy : 86.667%
Precision : 72.727%
Recall : 32.000%
F-1 score : 44.444%
Classification Report:
              precision    recall  f1-score   support

           0       0.88      0.98      0.92       500
           1       0.73      0.32      0.44       100

    accuracy                           0.87       600
   macro avg       0.80      0.65      0.68       600
weighted avg       0.85      0.87      0.84       600

